# ⚡ Chapter 13: Serverless Technologies

📖 Book: Chapter 13, pp. 512–573 · ⏱️ 2–3 hours · 🧱 Needs: Ch 9, 12 · Level: Advanced

---

> **🎯 Key idea:** FaaS (AWS Lambda, Cloud Run functions, Azure Functions v2) scales to zero per event, requiring idempotency and JSON proxy serialization.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** Python 3.7, CDK v1 (`@aws-cdk/*`)  
> **🆕 Now (2026):** Python 3.12/3.13, CDK v2 (`aws-cdk-lib`) / SAM  
> **Why:** EOL runtimes & unified CDK v2

> **📜 Then (book, ~2020):** Cloud Functions, Azure v1 `function.json`  
> **🆕 Now (2026):** Cloud Run functions, Azure v2 `@app.route`  
> **Why:** Unified GCP runtime & Pythonic Azure

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
Your webhook endpoint gets 10,000 hits during a 5-minute launch and zero traffic for the next 12 hours—paying for idle VMs wastes budget. Yet moving to serverless functions introduces cold starts, duplicate event deliveries, and API Gateway formatting rules. This chapter shows how to build idempotent, secure FaaS handlers and CDK v2 stacks.

### Concept: FaaS vs. CaaS & Cold Starts
**What:** Event functions vs. serverless containers  
**Why:** FaaS has 15-min max; CaaS runs heavy/long jobs  
**How:** Init SDK clients outside `lambda_handler` for reuse  
**When:** FaaS for APIs/webhooks; CaaS for ML

📖 *Book listing:* Multi-Cloud `currentTime` Handler & OpenFaaS (pp. 518–544)

**Plain-language analogy:** FaaS (Lambda) is like hailing a taxi only for the exact minutes you ride; CaaS (Fargate) is renting a car when you have heavy luggage or a long trip.

**Walkthrough:** The book deploys the same Python time function to AWS Lambda, GCP, Azure, and self-hosted OpenFaaS on `k3s`, highlighting stateless JSON responses and cold-start reuse.

### Concept: Idempotency & `PAY_PER_REQUEST`
**What:** Safe retries & auto-scaled DynamoDB  
**Why:** Duplicate events & traffic spikes break naive code  
**How:** Use upserts/dedup IDs + `PAY_PER_REQUEST` billing  
**When:** All event-driven serverless APIs

📖 *Book listing:* AWS CDK DynamoDB + Lambda Todo API (pp. 545–570)

**Plain-language analogy:** Idempotency is like an elevator button: pressing it five times still calls the elevator once without breaking anything.

**Walkthrough:** The book defines a DynamoDB table, 5 CRUD Lambdas, and an API Gateway in Python CDK, then discovers read-capacity throttling under Taurus/Molotov load and switches to `PAY_PER_REQUEST`.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: API Gateway proxy response helper + UTC time handler
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: API Gateway proxy response helper + UTC time handler
import json
from datetime import datetime, timezone

def api_response(status_code: int, payload: dict) -> dict:
    return {'statusCode': status_code, 'headers': {'Content-Type': 'application/json'}, 'body': json.dumps(payload)}

def lambda_handler(event: dict, context=None) -> dict:
    now = datetime.now(timezone.utc).isoformat()
    return api_response(200, {'utc_time': now, 'path': event.get('path', '/')})

print(lambda_handler({'path': '/time'}))

{'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"utc_time": "2026-11-27T09:05:16.205526+00:00", "path": "/time"}'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Query parameter validation
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Query parameter validation
def greet_handler(event: dict, context=None) -> dict:
    params = event.get('queryStringParameters') or {}
    name = params.get('name')
    if not name:
        return api_response(400, {'error': 'Missing query parameter: name'})
    return api_response(200, {'greeting': f'Hello, {name}!'})

print(greet_handler({'queryStringParameters': {'name': 'DevOps'}}))
print(greet_handler({}))

{'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"greeting": "Hello, DevOps!"}'}
{'statusCode': 400, 'headers': {'Content-Type': 'application/json'}, 'body': '{"error": "Missing query parameter: name"}'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Warm module-level connection pool pattern
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Warm module-level connection pool pattern
SHARED_POOL = {'initialized': True, 'region': 'us-east-1'}

def warm_handler(event: dict, context=None) -> dict:
    return api_response(200, {'reused_pool': SHARED_POOL['initialized']})

print(warm_handler({}))

{'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"reused_pool": true}'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Idempotent CRUD Simulator & AWS CDK v2

#### Example 4: Intermediate Example 1: Idempotent Todo CRUD router
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Idempotent Todo CRUD router
import json

TABLE: dict[str, dict] = {}

def todo_handler(event: dict, context=None) -> dict:
    method = event.get('httpMethod', 'GET')
    if method == 'POST':
        body = json.loads(event.get('body') or '{}')
        item_id = body.get('id')
        if not item_id:
            return api_response(400, {'error': 'Missing id'})
        TABLE[item_id] = {'id': item_id, 'task': body.get('task', '')}
        return api_response(201, TABLE[item_id])
    if method == 'GET':
        return api_response(200, {'items': list(TABLE.values())})
    return api_response(405, {'error': 'Method not allowed'})

print('POST:', todo_handler({'httpMethod': 'POST', 'body': '{"id": "1", "task": "Deploy"}'}))
print('GET:', todo_handler({'httpMethod': 'GET'}))

POST: {'statusCode': 201, 'headers': {'Content-Type': 'application/json'}, 'body': '{"id": "1", "task": "Deploy"}'}
GET: {'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"items": [{"id": "1", "task": "Deploy"}]}'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

> **⚠️ Warning:** Cloud commands incur costs—clean up test resources afterward.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): AWS CDK v2 (aws-cdk-lib)
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): AWS CDK v2 (aws-cdk-lib)
# Requires: pip install aws-cdk-lib constructs
# WARNING: Deploying CDK stacks creates real AWS resources. Run `cdk destroy` afterward!
try:
    from aws_cdk import aws_dynamodb
    print('aws-cdk-lib available, PAY_PER_REQUEST =', aws_dynamodb.BillingMode.PAY_PER_REQUEST)
except ImportError:
    cdk_spec = {'table': 'TodoTable', 'billing_mode': 'PAY_PER_REQUEST', 'runtime': 'PYTHON_3_12'}
    print('aws-cdk-lib not installed; simulated CDK v2 spec:', cdk_spec)

aws-cdk-lib not installed; simulated CDK v2 spec: {'table': 'TodoTable', 'billing_mode': 'PAY_PER_REQUEST', 'runtime': 'PYTHON_3_12'}


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **HMAC-SHA256 Webhook Verifier** using constant-time `hmac.compare_digest()`.

#### Example 6: import hashlib, hmac, json
*Expected output:* Printed labelled output below.

In [6]:
import hashlib, hmac, json

WEBHOOK_SECRET = b'devops-webhook-secret'

def verify_webhook(raw_body: str, signature: str) -> bool:
    expected = hmac.new(WEBHOOK_SECRET, raw_body.encode('utf-8'), hashlib.sha256).hexdigest()
    return hmac.compare_digest(signature, expected)

sample_body = '{"event": "push"}'
sig = hmac.new(WEBHOOK_SECRET, sample_body.encode(), hashlib.sha256).hexdigest()
print('Valid sig:', verify_webhook(sample_body, sig), '| Bad sig:', verify_webhook(sample_body, 'bad'))

Valid sig: True | Bad sig: False


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: sam CLI or py_compile check on Lambda handler
import py_compile, shutil, subprocess, tempfile
from pathlib import Path
if shutil.which('sam'):
    print('SAM CLI:', subprocess.run(['sam', '--version'], capture_output=True, text=True).stdout.strip())
with tempfile.TemporaryDirectory() as tmp:
    hp = Path(tmp) / 'handler.py'
    hp.write_text('def lambda_handler(event, ctx): return {"statusCode": 200}\n', encoding='utf-8')
    py_compile.compile(str(hp), doraise=True)
    print('Lambda handler syntax check PASSED')

Lambda handler syntax check PASSED


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Returning raw dict as `body`** -> `502 Bad Gateway` in API Gateway. ✅ Serialize with `json.dumps(payload)`.
- ❌ **Fixed `read_capacity=10`** -> `ProvisionedThroughputExceededException`. ✅ Use `BillingMode.PAY_PER_REQUEST`.
- ❌ **Relying on `/tmp` across calls** -> Missing state on cold starts. ✅ Persist state in DynamoDB or S3.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement extract_bearer_token and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement extract_bearer_token and run to verify!)
def _run_starter_checks():
    assert extract_bearer_token({'Authorization': 'Bearer abc'}) == 'abc'
def extract_bearer_token(headers: dict) -> str | None:
    raise NotImplementedError('Implement extract_bearer_token')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement extract_bearer_token


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** When choose CaaS (Fargate/Cloud Run) over FaaS (Lambda)?

<details>
<summary>✅ Answer 1</summary>

When tasks exceed FaaS timeouts (15 min), need custom binaries, or have heavy ML dependencies.

</details>

**Q2:** Why must API Gateway Lambda responses serialize `body` with `json.dumps()`?

<details>
<summary>✅ Answer 2</summary>

API Gateway proxy integration expects `body` to be a JSON string, not a Python dict.

</details>

**Q3:** What does `BillingMode.PAY_PER_REQUEST` solve in DynamoDB?

<details>
<summary>✅ Answer 3</summary>

Eliminates manual capacity provisioning and prevents throttling spikes.

</details>

**Q4:** Why initialize SDK clients outside `lambda_handler`?

<details>
<summary>✅ Answer 4</summary>

Warm containers reuse module-level connections, lowering latency.

</details>

**Q5:** What is idempotency in serverless?

<details>
<summary>✅ Answer 5</summary>

Duplicate event deliveries produce the exact same final state without duplicate side effects.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 3**: Check `auth.startswith('Bearer ')` and split on space.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
import json

def api_error(status: int, msg: str) -> dict:
    return api_response(status, {'error': msg})

def exceeds_lambda_limit(payload: dict) -> bool:
    return len(json.dumps(payload).encode('utf-8')) > 6 * 1024 * 1024

def extract_bearer_token(headers: dict) -> str | None:
    auth = headers.get('Authorization', '')
    return auth.split(' ', 1)[1] if auth.startswith('Bearer ') else None

def invoke_batch(handler, events: list[dict]) -> dict:
    ok = err = 0
    for ev in events:
        code_ = handler(ev)['statusCode']
        if 200 <= code_ < 300:
            ok += 1
        else:
            err += 1
    return {'2xx': ok, 'errors': err}

def webhook_handler(event: dict, context=None) -> dict:
    raw = event.get('body', '')
    sig = event.get('headers', {}).get('X-Signature', '')
    if not verify_webhook(raw, sig):
        return api_error(401, 'Invalid signature')
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return api_error(400, 'Malformed JSON')
    return api_response(202, {'accepted': True, 'event': data.get('event')})

assert exceeds_lambda_limit({'ok': True}) is False
assert extract_bearer_token({'Authorization': 'Bearer abc'}) == 'abc'
assert invoke_batch(greet_handler, [{'queryStringParameters': {'name': 'A'}}, {}]) == {'2xx': 1, 'errors': 1}
assert webhook_handler({'body': sample_body, 'headers': {'X-Signature': sig}})['statusCode'] == 202
assert webhook_handler({'body': sample_body, 'headers': {'X-Signature': 'bad'}})['statusCode'] == 401
print('All Chapter 13 assertions PASSED!')

All Chapter 13 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `lambda_handler(event, ctx)` — AWS FaaS entry point (e.g. `{'statusCode': 200, 'body': json.dumps(...)}`)
- `hmac.compare_digest()` — Constant-time webhook check (e.g. `verify_webhook(body, sig)`)
- `cdk deploy` / `cdk destroy` — Manage CDK v2 stacks (e.g. `cdk synth && cdk deploy`)

### ✅ What You Can Now Do
- Write API Gateway proxy-compatible Lambda handlers (`json.dumps(body)`).
- Implement idempotent CRUD routing and constant-time HMAC webhook verification.
- Understand Lambda Python 3.12+, AWS CDK v2 (`PAY_PER_REQUEST`), Cloud Run functions, and Azure Functions v2.

### ➡️ Next Chapter
Chapter 14 builds on this by applying containers, Kubernetes, and automated gates to Machine Learning (MLOps).